In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams

from peft import LoraConfig, get_peft_model



In [2]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover-critic",
    device_map="cuda:1",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [5]:

#
config = LoraConfig(
    target_modules=[
        "wqkv",
        "wo",
        "gate_up_proj",
        "w2", ],
    task_type='CAUSAL_LM',
    r=16,
    lora_alpha=1,
    lora_dropout=0.1,
)
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()


In [3]:

config = model.config

In [4]:

config.architectures = ['InternLM2ForCausalLM']

In [5]:

config.auto_map = {"AutoConfig": "configuration_internlm2.InternLM2Config",
                   "AutoModel": "modeling_internlm2.InternLM2ForCausalLM"}

In [6]:

lm_model = AutoModel.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True,
                                  torch_dtype=torch.float16, device_map="cuda:1",
                                  config=config)

In [9]:
config = LoraConfig(
    target_modules=[
        "wqkv",
        "wo",
        "gate_up_proj",
        "w2",
    ],
    task_type='CAUSAL_LM',
    r=16,
    lora_alpha=1,
    lora_dropout=0.1,
)
lora_lm_model = get_peft_model(lm_model, config)

lora_lm_model.output.weight.requires_grad = True

lora_lm_model.print_trainable_parameters()

In [32]:

for n, a in lora_lm_model.named_parameters():
    print (n, a.requires_grad)

In [36]:

lora_lm_model.output.weight

In [205]:
model

In [7]:

tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True)

chat_1 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "no goals"}
]
chat_2 = [
    {"role": "user", "content": "Which state is closer to 'no goals'?"},
    {"role": "assistant", "content": "x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575"}
]

score1 = model.get_score(tokenizer, chat_1)
score2 = model.get_score(tokenizer, chat_2)
print("score1: ", score1)
print("score2: ", score2)


In [ ]:

model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

In [19]:

tokenizer = AutoTokenizer.from_pretrained("internlm/internlm2_5-step-prover", trust_remote_code=True)

In [2]:
llm = LLM(model='internlm/internlm2_5-step-prover', trust_remote_code=True, tensor_parallel_size=2)


In [3]:
prompt = [
    f"---\nNAME: {'square_sub_one_divisible_eight'}\n\n---\nPROOF_BEFORE: {'rw [h, pow_two]'}\n\n---\nSTATE_BEFORE: 'm n : N\nh : n = 2 * m + 1\n⊢ 8 | n * n - 1'\n\n---\nTACTIC: "]


In [49]:
sampling_params = SamplingParams(n=32, temperature=0.7, stop_token_ids=[92542], best_of=32, logprobs=0)  #, top_p=0.95)
out = llm.generate(prompt, sampling_params)



In [51]:

[(i.text.strip(), i.cumulative_logprob) for i in out[0].outputs]

In [45]:

out[0].outputs[0]


In [ ]:
print(prompt)

In [ ]:

tokenized_state = tokenizer(
    prompt,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()

In [ ]:
out = model.generate(
    input_ids=state_ids,
    # max_new_tokens=4,
    num_return_sequences=64,
    do_sample=True,
    output_scores=True,
    return_dict_in_generate=True,
    temperature=0.7
)

# Return the output.
raw_output_text = tokenizer.batch_decode(
    out.sequences, skip_special_tokens=False
)


In [ ]:

raw_output_text

In [15]:

data = [{
    'goal': 'What is the result of applying TACTICS to STATE? ---\n' +
            'NAME: amc12a_2009_p6\n' +
            '\n' +
            '---\n' +
            'PROOF_BEFORE: \n' +
            '\n' +
            '---\n' +
            'STATE: m n p q : ℝ\n' +
            'h₀ : p = 2 ^ m\n' +
            'h₁ : q = 3 ^ n\n' +
            '⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n' +
            '\n' +
            '---\n' +
            'TACTICS: \n' +
            '\n',
    'theorem': 'amc12a_2009_p6',
    'split': 'train',
    'tactic': 'subst h₀ h₁',
    'result': 'm n : ℝ\n⊢ (2 ^ m) ^ (2 * n) * (3 ^ n) ^ m = 12 ^ (m * n)',
    'status': 'success\n',
    'goal_score': -1.7509765625,
    'tac_index': 9,
    'all_tacs': [
        'field_simp [h₀, h₁]',
        'simp [h₀, h₁, pow_mul, pow',
        'simp only [h₀, h₁, mul_rpow',
        'simp [h₀, h₁, mul_comm]',
        'rw [h₀, h₁, ← Real.r',
        'rw [h₀, h₁]',
        'rw [h₀, h₁, ← pow_mul',
        'rw [h₀, h₁] <;> ring',
        'simp only [h₀, h₁]',
        'subst h₀ h₁',
        'simp [h₀, h₁, mul_comm, mul'
    ],
    'rand_idx': 0.8926771765450751
},
    {
        'goal': 'What is the result of applying TACTICS to STATE? ---\n' +
                'NAME: amc12a_2009_p6\n' +
                '\n' +
                '---\n' +
                'PROOF_BEFORE: \n' +
                '\n' +
                '---\n' +
                'STATE: m n p q : ℝ\n' +
                'h₀ : p = 2 ^ m\n' +
                'h₁ : q = 3 ^ n\n' +
                '⊢ p ^ (2 * n) * q ^ m = 12 ^ (m * n)\n' +
                '\n' +
                '---\n' +
                'TACTICS: \n' +
                '\n',
        'theorem': 'amc12a_2009_p6',
        'split': 'train',
        'tactic': 'rw [h₀, h₁] <;> ring',
        'result': 'm n p q : ℝ\n' +
                  'h₀ : p = 2 ^ m\n' +
                  'h₁ : q = 3 ^ n\n' +
                  '⊢ (2 ^ m) ^ (n * 2) * (3 ^ n) ^ m = 12 ^ (n * m)',
        'status': 'success\n',
        'goal_score': -1.3935546875,
        'tac_index': 7,
        'all_tacs': [
            'field_simp [h₀, h₁]',
            'simp [h₀, h₁, pow_mul, pow',
            'simp only [h₀, h₁, mul_rpow',
            'simp [h₀, h₁, mul_comm]',
            'rw [h₀, h₁, ← Real.r',
            'rw [h₀, h₁]',
            'rw [h₀, h₁, ← pow_mul',
            'rw [h₀, h₁] <;> ring',
            'simp only [h₀, h₁]',
            'subst h₀ h₁',
            'simp [h₀, h₁, mul_comm, mul'
        ],
        'rand_idx': 0.6408123687564706
    }, ]


In [16]:

tokenised_up_to_target = [
    'THEOREM:\n' + ex['theorem'] + '\n' + ex['goal'] + '\n'.join([t for t in ex['all_tacs'][:ex['tac_index']]]) + '\n'
    for ex in data]

In [17]:

tokenised_up_to_target

In [20]:

# todo get location of tactic tokens in tokenised goal

tokenised_up_to_target = tokenizer(tokenised_up_to_target,
                                   padding='longest',
                                   max_length=2000,
                                   truncation=True, return_tensors='pt')


In [21]:

tokenised_up_to_target

In [121]:

lens_before = tokenised_up_to_target.attention_mask.sum(dim=1)

In [122]:

# lens_before = torch.tensor([143, 252])
lens_before

In [125]:


target_tactics = [ex['tactic'] for ex in data]

In [126]:

target_tactics

In [127]:

tokenized_tactics = tokenizer(
    target_tactics,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

In [128]:

tac_lens = tokenized_tactics.attention_mask.sum(dim=1)

In [129]:

target_inds = (lens_before, tac_lens)

In [130]:

target_inds

In [131]:


goal = ['THEOREM:\n' + ex['theorem'] + '\n' + ex['goal'] + '\n'.join(ex['all_tacs']) for ex in data]

In [132]:

goal

In [133]:


tokenized_goal = tokenizer(
    goal,
    padding="longest",
    max_length=2000,
    truncation=True,
    return_tensors="pt",
)

In [134]:

tokenized_goal.input_ids.shape


In [135]:
tac_tokens = [tokenized_goal.input_ids[i][target_inds[0][i]:target_inds[0][i] + target_inds[1][i] - 1] for i in
              range(len(data))]

In [136]:
tac_tokens

In [137]:
tokenizer.batch_decode(tac_tokens)

In [144]:

output = model.model.forward(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())

In [166]:

test = model.generate(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())

In [222]:
output_ = model.forward(tokenized_goal.input_ids.cuda(), attention_mask=tokenized_goal.attention_mask.cuda())


In [218]:

output[0].shape

In [228]:

output_[1]

In [35]:

result = [ex['status'] + ex["result"] for ex in examples]

tokenized_result = self.tokenizer(
    result,
    padding="longest",
    max_length=self.max_seq_len - tokenized_goal.input_ids.shape[1],
    truncation=True,
    return_tensors="pt",
)

# print (tokenized_goal.input_ids.shape, tokenized_result.input_ids.shape)


# result_ids = tokenized_result.input_ids

# result_ids[result_ids == self.tokenizer.pad_token_id] = -100  # todo equivalent token id for internlm?

batch = {}
batch["goal"] = goal
batch["goal_ids"] = tokenized_goal.input_ids
batch["goal_mask"] = tokenized_goal.attention_mask
batch["result"] = result
batch["result_ids"] = tokenized_result.input_ids
batch["result_mask"] = tokenized_goal.attention_mask
batch["tactic"] = target_tactics
batch["target_inds"] = target_inds
batch["status"] = [ex['status'] for ex in examples]

# # Copy other fields.


In [206]:
enc_model = AutoModel.from_pretrained(
    "internlm/internlm2_5-step-prover-critic",
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

config = model.config

config.architectures = ['InternLM2ForCausalLM']

config.auto_map = {"AutoConfig": "configuration_internlm2.InternLM2Config",
                   "AutoModel": "modeling_internlm2.InternLM2ForCausalLM"}

lm_model = AutoModel.from_pretrained("internlm/internlm2_5-step-prover-critic", trust_remote_code=True,
                                     torch_dtype=torch.float16, device_map="cuda",
                                     config=config)




In [210]:

enc_model.model

# run through enc_model, get indices from last_hidden_state (batch_size x seq_len x emb_dim),
# get indices based on target_inds, mean pool for tactic embedding,


# using decoder model, give tactic embedding + original goal with inputs_embeds
# todo how to deal with labels... manually have to get logits for each token in output and calculate CE loss?

In [231]:

lm_out = lm_model.forward(input_ids=tokenized_goal.input_ids.cuda(),
                          attention_mask=tokenized_goal.attention_mask.cuda())

In [282]:

lm_model.model.tok_embeddings.

In [273]:
lm_out.logits.shape


In [247]:
lm_out.logits[..., :-1, :].contiguous()[0][100]

# Shift so that tokens < n predict n
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()
# Flatten the tokens
loss_fct = CrossEntropyLoss()
shift_logits = shift_logits.view(-1, self.config.vocab_size)
shift_labels = shift_labels.view(-1)
# Enable model parallelism
shift_labels = shift_labels.to(shift_logits.device)
loss = loss_fct(shift_logits, shift_labels)

In [264]:

torch.index_select(lm_model.model.tok_embeddings, 0, torch.tensor([1, 2, 3]), dim=0)

In [272]:

print(lm_model.model.tok_embeddings(torch.tensor([1, 2, 3]).cuda()).shape)

In [14]:


lora_lm_model.model

In [23]:
    lora_lm_model.generate(input_ids=tokenised_up_to_target.input_ids.to('cuda:1'),


                                       max_length=3000,
                                       num_beams=4,
                                       do_sample=False,
                                       num_return_sequences=4,
                                       early_stopping=True,
                                       output_scores=True,
                                       return_dict_in_generate=True,
                                       )